# Speed-up stages on Colab L4 (plan: `worklog/plans/2026-11-20-speedup-plan.md`)
Open on colab.research.google.com; **Runtime -> Change runtime type -> L4 GPU**. Never paste the HF token into a cell.

- **tune** = `train-00000` (all tuning), **holdout** = `train-00024` + `train-00049` (scored only, never tuned on).
- References and logs live on Drive: `DRIVE=/content/drive/MyDrive/vidx` (`reference/`, `frames/`, `bench_log.csv`).
- Accept bar (tune AND holdout): F1 >= 0.985, people exact >= 0.93, within +-1 >= 0.985.
- Fallback ladder, fixed in advance: k=3 -> k=5 -> every frame fp16 -> every frame fp32.

**After every runtime reset:** run section 0 (all of it), then the step you were on. Long runs: wait for them to finish
(`tail` the log) before running anything that reads their output.

## 0. Setup (re-run after every reset)
### 0.1 Machine, venv, code, model

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; free -h | head -2; df -h /content | tail -1
cd /content
pip install -q uv
[ -d /content/venv ] || uv venv --python 3.11 /content/venv
[ -d video-indexing-yolo11 ] || git clone https://github.com/sobanali256/video-indexing-yolo11.git
cd video-indexing-yolo11 && git pull -q && git log --oneline -1
apt-get install -y -q libgl1 libglib2.0-0 > /dev/null
uv pip install -q --python /content/venv/bin/python -r requirements.txt
/content/venv/bin/python -c "import vidx,torch,av,ultralytics;print(torch.__version__,torch.cuda.is_available(),av.__version__,ultralytics.__version__)"
/content/venv/bin/python prepare.py

### 0.2 Hugging Face login (Colab Secret `HF_TOKEN`, else prompt)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    import getpass
    login(getpass.getpass("HF token: "))

### 0.3 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!ls /content/drive/MyDrive/vidx /content/drive/MyDrive/vidx/reference

### 0.4 Start the 50-file download in the background (skips files already complete)

In [ ]:
!pgrep -f "hf download" > /dev/null && echo "download already running" || (nohup /content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir /content/finevideo > /content/download.log 2>&1 &)
!sleep 5; tail -c 300 /content/download.log; echo; ls /content/finevideo/data 2>/dev/null | wc -l

### 0.5 Download status + tune/holdout links (re-run until the 3 reference files are there)

In [ ]:
%%bash
tail -c 200 /content/download.log; echo
D=/content/finevideo/data
echo "parquet files: $(ls $D/*.parquet 2>/dev/null | wc -l) / 50, size $(du -sh /content/finevideo 2>/dev/null | cut -f1)"
mkdir -p /content/subset/tune /content/subset/holdout
for f in $D/train-00000-*.parquet; do [ -e "$f" ] && ln -sf "$f" /content/subset/tune/; done
for f in $D/train-00024-*.parquet $D/train-00049-*.parquet; do [ -e "$f" ] && ln -sf "$f" /content/subset/holdout/; done
ls -l /content/subset/tune /content/subset/holdout

## 1. Stage 1a: sampling simulator (CPU only, needs only Drive)
The `check k=0` line must be 1.0000. Results 2026-11-20: k=3 accepted (F1 0.9880, exact 0.9584, within1 0.9988).

In [ ]:
%%bash
cd /content/video-indexing-yolo11
D=/content/drive/MyDrive/vidx
echo "dumps: $(ls $D/frames | wc -l) (expect 33)"
/content/venv/bin/python tools/simulate_sampling.py --dumps $D/frames --ref $D/reference/ref_tune.jsonl   --k 1,2,3,4,5,6,7,9 --adaptive 3:7,3:9,5:9 --json $D/sim_tune.json

## 2. Stage 0: profile (needs the tune file). Done 2026-11-20
fp32 99 fps, fp16 155 fps (accuracy 0.9995 / 0.998 / 1.0); decode ~0.2 ms/frame. `fail_accuracy` here is expected (`--only-common` = PARTIAL).

In [ ]:
%%bash
cd /content/video-indexing-yolo11
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python
$PY tools/profile_decode.py --input /content/subset/tune --videos 5
for P in fp32 fp16; do
  $PY bench.py run --label "profile $P every frame" --input /content/subset/tune --out /content/prof_$P.jsonl     --ref tune=$D/reference/ref_tune.jsonl --only-common --log $D/bench_log.csv -- --limit-videos 5 --precision $P     2>&1 | grep -E "^(PASS|FAIL|PARTIAL|bench|videos)"
done
tail -2 $D/bench_log.csv

## 3. Stage 1 gate: k=3 + FP16
1. k=3 FP32 on tune must reproduce the simulator exactly (F1 0.9880, exact 0.9584, within1 0.9988): proves the decoder samples the simulated frames.
2. k=3 FP16 on tune: accept bar.
3. k=3 FP16 on holdout: accept bar (holdout is never used to choose anything; if it fails, step down the ladder).

In [ ]:
%%bash
cd /content/video-indexing-yolo11 && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
run() {  # label input out ref -- run.py args
  $PY bench.py run --label "$1" --input "$2" --out "$3" --ref "$4" --log $D/bench_log.csv -- "${@:5}"     2>&1 | grep -E "^(PASS|FAIL|bench|videos|overall)"
  cp "$3" $D/
}
run "stage1 k3 fp32 tune"    /content/subset/tune    /content/s1_k3_fp32_tune.jsonl    tune=$R/ref_tune.jsonl       --sample 3 --precision fp32
run "stage1 k3 fp16 tune"    /content/subset/tune    /content/s1_k3_fp16_tune.jsonl    tune=$R/ref_tune.jsonl       --sample 3 --precision fp16
run "stage1 k3 fp16 holdout" /content/subset/holdout /content/s1_k3_fp16_holdout.jsonl holdout=$R/ref_holdout.jsonl --sample 3 --precision fp16
tail -3 $D/bench_log.csv

## Helpers

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv; pgrep -af "run.py|hf download" | cut -c1-150; tail -c 300 /content/download.log